# Combined Sparse Features — Quick Search

Compares **word TF-IDF**, **character TF-IDF**, **numeric features**, and their **combination**
on the same DVC train/validation split. Evaluates preprocessing (lowercase, lemmatization,
stop-word strategies) and five classifier families.

Designed to finish in roughly **10–20 minutes** (no XGBoost in the grid).

In [ ]:
import json
import re
import warnings
from datetime import datetime
from itertools import product
from pathlib import Path
from time import perf_counter

import joblib
import mlflow
import nltk
import numpy as np
import pandas as pd
from nltk.stem import WordNetLemmatizer
from scipy.sparse import csr_matrix, hstack
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier

from helpers.data import load_processed_data, load_sample_submission
from helpers.metrics import compute_all_metrics
from helpers.mlflow import log_common_context, log_metrics, log_model_artifacts, start_notebook_run
from helpers.submission import build_submission, save_submission

warnings.filterwarnings('ignore', category=FutureWarning)

WORD_MAX_FEATURES = 10_000
CHAR_MAX_FEATURES = 8_000
WORD_NGRAM_RANGE = (1, 2)
CHAR_NGRAM_RANGE = (2, 4)

MODEL_FAMILIES = [
    'logistic_regression',
    'linear_svc',
    'naive_bayes',
    'decision_tree',
    'random_forest',
]


def log_step(message: str) -> None:
    print(f'[{datetime.now():%H:%M:%S}] {message}')

## Load Data

In [ ]:
PROCESSED_DIR = Path('../data/processed')
RAW_DIR = Path('../data/raw')
KAGGLE_INPUT_DIR = Path('/kaggle/input/competitions/contradictory-my-dear-watson')

if (PROCESSED_DIR / 'train_split.csv').exists():
    train_df, val_df, test_df = load_processed_data(PROCESSED_DIR)
    sample_submission = load_sample_submission(RAW_DIR / 'sample_submission.csv')
    data_source = 'dvc_processed_split'
elif (KAGGLE_INPUT_DIR / 'train.csv').exists():
    full_train_df = pd.read_csv(KAGGLE_INPUT_DIR / 'train.csv')
    test_df = pd.read_csv(KAGGLE_INPUT_DIR / 'test.csv')
    sample_submission = pd.read_csv(KAGGLE_INPUT_DIR / 'sample_submission.csv')
    train_df, val_df = train_test_split(
        full_train_df,
        test_size=0.2,
        random_state=42,
        stratify=full_train_df['label'],
    )
    data_source = 'kaggle_input_fallback_split'
else:
    raise FileNotFoundError('Could not find DVC processed splits or Kaggle input files.')

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)
y_train = train_df['label'].astype(int)
LANGUAGE_COLUMNS = sorted(train_df['lang_abv'].dropna().unique().tolist())

log_step(f'Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}, langs={len(LANGUAGE_COLUMNS)}')

## Feature Builders

| Group | Content |
|-------|---------|
| **word_tfidf** | Word n-grams on premise + hypothesis |
| **char_tfidf** | Char n-grams (normalized text) |
| **numeric** | Lengths, word counts, language one-hot |
| **combined** | Horizontal stack of all three |

In [ ]:
PUNCTUATION_PATTERN = re.compile(r"[^\w\s]", re.UNICODE)
NEGATION_WORDS = {'not', 'no', 'never', 'neither', 'nor', 'none', 'cannot'}
SAFE_ENGLISH_STOP_WORDS = {w for w in ENGLISH_STOP_WORDS if w not in NEGATION_WORDS}


def ensure_nltk_resources() -> None:
    for resource in ('corpora/wordnet', 'corpora/omw-1.4'):
        try:
            nltk.data.find(resource)
        except LookupError:
            nltk.download(resource.split('/')[-1], quiet=True)


ensure_nltk_resources()
LEMMATIZER = WordNetLemmatizer()


def normalize_char_text(series: pd.Series) -> pd.Series:
    texts = series.fillna('').astype(str).str.lower()
    texts = texts.str.replace(PUNCTUATION_PATTERN, ' ', regex=True)
    return texts.str.split().str.join(' ')


def preprocess_word_text(text: str, language: str, text_config: dict) -> str:
    value = str(text).lower() if text_config['lowercase'] else str(text)
    if text_config['lowercase']:
        value = PUNCTUATION_PATTERN.sub(' ', value)
    words = value.split()
    if text_config['lemmatize'] and language == 'en':
        words = [LEMMATIZER.lemmatize(word) for word in words]
    if text_config['stop_words'] == 'safe_english' and language == 'en':
        words = [word for word in words if word not in SAFE_ENGLISH_STOP_WORDS]
    return ' '.join(words)


def word_series(frame: pd.DataFrame, column: str, text_config: dict) -> pd.Series:
    return frame.apply(
        lambda row: preprocess_word_text(row[column], row['lang_abv'], text_config),
        axis=1,
    )


def fit_word_vectorizer(train_frame: pd.DataFrame, text_config: dict) -> TfidfVectorizer:
    vectorizer = TfidfVectorizer(
        ngram_range=WORD_NGRAM_RANGE,
        max_features=WORD_MAX_FEATURES,
        lowercase=False,
        sublinear_tf=True,
    )
    corpus = pd.concat([
        word_series(train_frame, 'premise', text_config),
        word_series(train_frame, 'hypothesis', text_config),
    ])
    vectorizer.fit(corpus)
    return vectorizer


def word_block(frame: pd.DataFrame, vectorizer: TfidfVectorizer, text_config: dict):
    premise = word_series(frame, 'premise', text_config)
    hypothesis = word_series(frame, 'hypothesis', text_config)
    return hstack([
        vectorizer.transform(premise),
        vectorizer.transform(hypothesis),
    ], format='csr')


def fit_char_vectorizer(train_frame: pd.DataFrame) -> TfidfVectorizer:
    vectorizer = TfidfVectorizer(
        analyzer='char',
        ngram_range=CHAR_NGRAM_RANGE,
        max_features=CHAR_MAX_FEATURES,
        lowercase=False,
        sublinear_tf=True,
    )
    corpus = pd.concat([
        normalize_char_text(train_frame['premise']),
        normalize_char_text(train_frame['hypothesis']),
    ])
    vectorizer.fit(corpus)
    return vectorizer


def char_block(frame: pd.DataFrame, vectorizer: TfidfVectorizer):
    premise = normalize_char_text(frame['premise'])
    hypothesis = normalize_char_text(frame['hypothesis'])
    return hstack([
        vectorizer.transform(premise),
        vectorizer.transform(hypothesis),
    ], format='csr')


def numeric_block(frame: pd.DataFrame) -> csr_matrix:
    premise = frame['premise'].fillna('').astype(str)
    hypothesis = frame['hypothesis'].fillna('').astype(str)
    prem_words = premise.str.split().str.len()
    hyp_words = hypothesis.str.split().str.len()
    numeric = pd.DataFrame({
        'prem_char_len': premise.str.len(),
        'hyp_char_len': hypothesis.str.len(),
        'prem_word_len': prem_words,
        'hyp_word_len': hyp_words,
        'char_len_diff': (premise.str.len() - hypothesis.str.len()).abs(),
        'word_len_diff': (prem_words - hyp_words).abs(),
    })
    language = (
        pd.get_dummies(frame['lang_abv'])
        .reindex(columns=LANGUAGE_COLUMNS, fill_value=0)
        .astype('float32')
    )
    return csr_matrix(pd.concat([numeric, language], axis=1))


def build_feature_bundle(train_frame, val_frame, experiment_config):
    text_config = experiment_config['text_config']
    mode = experiment_config['feature_mode']
    blocks = []
    artifacts = {'feature_mode': mode, 'text_config': text_config}

    if mode in {'word_only', 'combined'}:
        word_vectorizer = fit_word_vectorizer(train_frame, text_config)
        artifacts['word_vectorizer'] = word_vectorizer
        blocks.append(('word', word_block))
    if mode in {'char_only', 'combined'}:
        char_vectorizer = fit_char_vectorizer(train_frame)
        artifacts['char_vectorizer'] = char_vectorizer
        blocks.append(('char', char_block))
    if mode in {'numeric_only', 'combined'}:
        blocks.append(('numeric', lambda f, _: numeric_block(f)))

    def stack_split(frame):
        parts = []
        for name, builder in blocks:
            if name == 'word':
                parts.append(builder(frame, artifacts['word_vectorizer'], text_config))
            elif name == 'char':
                parts.append(builder(frame, artifacts['char_vectorizer']))
            else:
                parts.append(builder(frame, None))
        return hstack(parts, format='csr') if len(parts) > 1 else parts[0]

    return stack_split(train_frame), stack_split(val_frame), artifacts


def build_test_features(test_frame, artifacts, experiment_config):
    text_config = experiment_config['text_config']
    mode = experiment_config['feature_mode']
    parts = []
    if mode in {'word_only', 'combined'}:
        parts.append(word_block(test_frame, artifacts['word_vectorizer'], text_config))
    if mode in {'char_only', 'combined'}:
        parts.append(char_block(test_frame, artifacts['char_vectorizer']))
    if mode in {'numeric_only', 'combined'}:
        parts.append(numeric_block(test_frame))
    return hstack(parts, format='csr') if len(parts) > 1 else parts[0]


def make_classifier_pipeline(model_family: str) -> Pipeline:
    estimators = {
        'logistic_regression': LogisticRegression(
            C=1.0,
            max_iter=5000,
            solver='saga',
            random_state=42,
        ),
        'linear_svc': LinearSVC(random_state=42, max_iter=5000),
        'naive_bayes': MultinomialNB(),
        'decision_tree': DecisionTreeClassifier(max_depth=20, random_state=42),
        'random_forest': RandomForestClassifier(
            n_estimators=80,
            max_depth=20,
            random_state=42,
            n_jobs=-1,
        ),
    }
    return Pipeline([('clf', estimators[model_family])])

## Experiment Plan

**Individual baselines:** `word_only`, `char_only`, `numeric_only`

**Combined + preprocessing variants:** `combined` with keep-all / safe stops / lemmatization

**6 configs × 5 models = 30 runs** (test features built only for the production winner).

In [ ]:
TEXT_CONFIGS = {
    'norm_keep': {'lowercase': True, 'lemmatize': False, 'stop_words': 'keep_all'},
    'norm_safe_stop': {'lowercase': True, 'lemmatize': False, 'stop_words': 'safe_english'},
    'lemma_safe_stop': {'lowercase': True, 'lemmatize': True, 'stop_words': 'safe_english'},
}

EXPERIMENT_CONFIGS = [
    {'experiment_id': 'word_only__norm_keep', 'feature_mode': 'word_only', 'text_key': 'norm_keep'},
    {'experiment_id': 'char_only__baseline', 'feature_mode': 'char_only', 'text_key': 'norm_keep'},
    {'experiment_id': 'numeric_only__baseline', 'feature_mode': 'numeric_only', 'text_key': 'norm_keep'},
    {'experiment_id': 'combined__norm_keep', 'feature_mode': 'combined', 'text_key': 'norm_keep'},
    {'experiment_id': 'combined__norm_safe_stop', 'feature_mode': 'combined', 'text_key': 'norm_safe_stop'},
    {'experiment_id': 'combined__lemma_safe_stop', 'feature_mode': 'combined', 'text_key': 'lemma_safe_stop'},
]

for cfg in EXPERIMENT_CONFIGS:
    cfg['text_config'] = TEXT_CONFIGS[cfg['text_key']]

SEARCH_PLAN = list(product(EXPERIMENT_CONFIGS, MODEL_FAMILIES))
log_step(f'{len(SEARCH_PLAN)} runs planned')

## Grid Search

In [ ]:
feature_cache = {}
grid_results = []
grid_started = perf_counter()

for experiment_config, model_family in SEARCH_PLAN:
    experiment_id = experiment_config['experiment_id']
    run_label = f'{experiment_id}__{model_family}'
    run_started = perf_counter()

    if experiment_id not in feature_cache:
        t0 = perf_counter()
        x_train, x_val, artifacts = build_feature_bundle(train_df, val_df, experiment_config)
        feature_cache[experiment_id] = (x_train, x_val, artifacts)
        log_step(f'Features {experiment_id} in {perf_counter() - t0:.1f}s | shape={x_train.shape}')

    x_train, x_val, _ = feature_cache[experiment_id]
    pipeline = make_classifier_pipeline(model_family)
    pipeline.fit(x_train, y_train)
    metrics = compute_all_metrics(val_df, pipeline.predict(x_val))
    elapsed = perf_counter() - run_started

    grid_results.append({
        'run_label': run_label,
        'experiment_id': experiment_id,
        'feature_mode': experiment_config['feature_mode'],
        'text_key': experiment_config['text_key'],
        'lemmatize': experiment_config['text_config']['lemmatize'],
        'stop_words': experiment_config['text_config']['stop_words'],
        'model_family': model_family,
        'accuracy': metrics['accuracy'],
        'f1_macro': metrics['f1_macro'],
        'train_features': x_train.shape[1],
        'elapsed_seconds': round(elapsed, 1),
    })
    log_step(
        f'{run_label} acc={metrics["accuracy"]:.4f} f1={metrics["f1_macro"]:.4f} ({elapsed:.1f}s)'
    )

log_step(f'Grid done in {(perf_counter() - grid_started) / 60:.1f} min')
results_df = pd.DataFrame(grid_results).sort_values(['f1_macro', 'accuracy'], ascending=False)
results_df

## Combined vs Individual (best per group)

In [ ]:
def best_per_group(frame: pd.DataFrame, group_col: str) -> pd.DataFrame:
    return (
        frame.sort_values(['f1_macro', 'accuracy'], ascending=False)
        .groupby(group_col, as_index=False)
        .first()
    )

feature_group_comparison = best_per_group(results_df, 'feature_mode')[
    ['feature_mode', 'model_family', 'accuracy', 'f1_macro', 'experiment_id', 'elapsed_seconds']
]
preprocess_comparison = best_per_group(
    results_df[results_df['feature_mode'] == 'combined'],
    'text_key',
)[['text_key', 'lemmatize', 'stop_words', 'model_family', 'accuracy', 'f1_macro']]
model_comparison = best_per_group(results_df, 'model_family')[
    ['model_family', 'feature_mode', 'accuracy', 'f1_macro', 'elapsed_seconds']
]

print('Best run per feature group (individual vs combined):')
feature_group_comparison
print('Best combined runs per preprocessing/stop strategy:')
preprocess_comparison
print('Best run per model family:')
model_comparison

## Production — Best Combined Sparse + Model

Winner = highest validation macro-F1 among **combined** runs; if none beat the best individual
baseline, the overall top row is still saved but `feature_mode` in metadata documents the choice.

In [ ]:
combined_results = results_df[results_df['feature_mode'] == 'combined'].sort_values(
    ['f1_macro', 'accuracy'],
    ascending=False,
)
best_overall = results_df.iloc[0]
best_result = combined_results.iloc[0]
selection_note = 'best_combined_sparse_for_production'
combined_beats_overall = best_result['f1_macro'] >= best_overall['f1_macro'] - 1e-9

best_experiment_id = best_result['experiment_id']
best_model_family = best_result['model_family']
best_experiment_config = next(c for c in EXPERIMENT_CONFIGS if c['experiment_id'] == best_experiment_id)

log_step(f'Production: {best_result["run_label"]} ({selection_note})')

x_train, x_val, artifacts = build_feature_bundle(train_df, val_df, best_experiment_config)
x_test = build_test_features(test_df, artifacts, best_experiment_config)

production_pipeline = make_classifier_pipeline(best_model_family)
production_pipeline.fit(x_train, y_train)
production_metrics = compute_all_metrics(val_df, production_pipeline.predict(x_val))
test_preds = production_pipeline.predict(x_test)

RUN_NAME = f'combined_sparse_{best_experiment_id}__{best_model_family}'
submission_path = Path('../submissions') / f'{RUN_NAME}.csv'
save_submission(build_submission(sample_submission, test_preds), submission_path)

results_path = Path('../submissions') / 'combined_sparse_grid_results.csv'
results_df.to_csv(results_path, index=False)

bundle_path = Path('../submissions') / f'{RUN_NAME}_feature_bundle.joblib'
pipeline_path = Path('../submissions') / f'{RUN_NAME}_pipeline.joblib'
joblib.dump({'pipeline': production_pipeline, **artifacts}, bundle_path)
joblib.dump(production_pipeline, pipeline_path)

serving_type = 'sklearn_combined_sparse_features'
metadata = {
    'run_name': RUN_NAME,
    'data_source': data_source,
    'serving_type': serving_type,
    'selection_note': selection_note,
    'experiment': best_experiment_config,
    'model_family': best_model_family,
    'language_columns': LANGUAGE_COLUMNS,
    'feature_group_comparison': feature_group_comparison.to_dict(orient='records'),
    'best_grid_result': best_result.to_dict(),
}

with start_notebook_run(
    RUN_NAME,
    tags={'features': 'combined_sparse', 'stage': 'production_candidate'},
):
    mlflow.log_params({
        'feature_mode': best_experiment_config['feature_mode'],
        'text_key': best_experiment_config['text_key'],
        'model_family': best_model_family,
        'selection_note': selection_note,
        'data_source': data_source,
    })
    log_metrics(production_metrics)
    log_common_context('../configs/data_split.yaml', '../data/processed/split_metadata.json')
    mlflow.log_artifact(str(submission_path), artifact_path='submissions')
    mlflow.log_artifact(str(results_path), artifact_path='research')
    log_model_artifacts(
        artifacts={'feature_bundle.joblib': bundle_path, 'pipeline.joblib': pipeline_path},
        metadata=metadata,
        artifact_path='model',
    )

word_only_best_f1 = (
    feature_group_comparison.loc[
        feature_group_comparison['feature_mode'] == 'word_only',
        'f1_macro',
    ].iloc[0]
    if (feature_group_comparison['feature_mode'] == 'word_only').any()
    else np.nan
)
improvement_vs_word = best_result['f1_macro'] - word_only_best_f1

pd.Series({
    'run_name': RUN_NAME,
    'feature_mode': best_result['feature_mode'],
    'selection_note': selection_note,
    'combined_beats_overall_best': combined_beats_overall,
    'overall_best_f1': best_overall['f1_macro'],
    'accuracy': production_metrics['accuracy'],
    'f1_macro': production_metrics['f1_macro'],
    'combined_vs_best_word_f1_delta': improvement_vs_word,
    'submission_path': str(submission_path),
})